# 01 - Load Gemma E4B on Colab and store it on Google Drive

Phase 1 of the project plan. This notebook does three things:

1. Downloads the E4B checkpoint from Kaggle onto the Colab disk.
2. Copies it to Google Drive, so later sessions restore it from Drive instead of re-downloading.
3. (Optional, last section) loads it on the GPU as a smoke test.

**Before you run it**
- Runtime type: **a CPU runtime is enough for sections 1-7** (download, store, restore) and uses far fewer
  compute units. Switch to the A100 or RTX 6000 GPU (Runtime > Change runtime type) only for section 8.
- Left sidebar > key icon (Secrets) > add a secret named `KAGGLE_API_TOKEN` with your Kaggle token,
  and switch on **Notebook access**. The token is never written into this notebook.
- Drive space: the competition checkpoint is about **11.5 GB** and the unquantized QAT one about **15.9 GB**
  (27.4 GB for the default pair). Make sure your Drive has room.

Run the cells top to bottom. Every step is safe to re-run: files already copied are skipped.

## 1. Config

In [ ]:
# ---- edit this cell only ----
FETCH = ["competition", "qat_unquantized"]   # any of: "competition", "bf16", "qat_unquantized"

DRIVE_DIR    = "/content/drive/MyDrive/gemma4-e4b"   # permanent copy
LOCAL_CACHE  = "/content/kagglehub"                  # Kaggle download cache (wiped with the session)
LOCAL_MODELS = "/content/models"                     # working copy restored from Drive

CHECKPOINTS = {
    # What the competition grader serves (compressed-tensors, 4-bit weights, 16-bit activations).
    "competition":     dict(handle="google/gemma-4/other/gemma-4-e4b-it-qat-w4a16-ct", gb=11.5),
    # Plain bf16 instruction-tuned weights.
    "bf16":            dict(handle="google/gemma-4/transformers/gemma-4-e4b-it", gb=16.0),
    # QAT weights stored unquantized: closest trainable match to the served model.
    "qat_unquantized": dict(handle="google/gemma-4/transformers/gemma-4-e4b-it-qat-q4_0-unquantized", gb=15.9),
}
for k in FETCH:
    assert k in CHECKPOINTS, f"unknown checkpoint {k!r}"
print("will fetch:", {k: CHECKPOINTS[k]["handle"] for k in FETCH})
print("approx download size: %.1f GB" % sum(CHECKPOINTS[k]["gb"] for k in FETCH))

## 2. Machine check

In [ ]:
import shutil, subprocess
try:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print("No GPU on this runtime. That is fine for sections 1-7; section 8 needs a GPU.")
total, used, free = shutil.disk_usage("/content")
print("local disk free: %.0f GB" % (free / 1e9))

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs(DRIVE_DIR, exist_ok=True)
try:
    total, used, free = shutil.disk_usage("/content/drive")
    print("Drive: %.0f GB free of %.0f GB" % (free / 1e9, total / 1e9))
    need = sum(CHECKPOINTS[k]["gb"] for k in FETCH)
    if free / 1e9 < need * 1.05:
        print("WARNING: Drive may not have room for", FETCH, "- free space or fetch fewer checkpoints.")
except Exception as e:
    print("could not read Drive free space:", e)

## 4. Kaggle login (from the Colab secret)

The token is read from the secret and kept only in this session's environment.

In [ ]:
from google.colab import userdata
try:
    os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
except Exception as e:
    raise RuntimeError(
        "Secret KAGGLE_API_TOKEN not available. Add it in the key icon (left sidebar) "
        "and turn on Notebook access, then re-run this cell.") from e
print("Kaggle token loaded (not shown).")

## 4b. Helper functions

Defines the copy, verify, store and restore helpers. **In a later session, run sections 1, 3, 4b and then 7;
no Kaggle login or download is needed.**

Files are copied one at a time to a `.part` name and renamed when finished, so an interrupted copy never
leaves a half-written file that looks complete. Re-running skips files whose size already matches.

In [ ]:
import json, os, shutil, time
from pathlib import Path


def dir_size(p):
    return sum(f.stat().st_size for f in Path(p).rglob("*") if f.is_file())


def files_of(root):
    root = Path(root)
    return {str(f.relative_to(root)): f.stat().st_size
            for f in root.rglob("*") if f.is_file() and not f.name.endswith(".part")}


def copy_tree(src, dst, retries=3):
    """Copy src -> dst, skipping files that already match by size. Safe to re-run."""
    src, dst = Path(src), Path(dst)
    for rel, size in sorted(files_of(src).items()):
        out = dst / rel
        if out.exists() and out.stat().st_size == size:
            continue
        out.parent.mkdir(parents=True, exist_ok=True)
        tmp = out.with_name(out.name + ".part")
        for attempt in range(1, retries + 1):
            try:
                t0 = time.time()
                shutil.copyfile(src / rel, tmp)
                os.replace(tmp, out)
                print("  copied %-45s %7.2f GB  %.0fs" % (rel, size / 1e9, time.time() - t0))
                break
            except OSError as e:
                print("  retry %d/%d for %s: %s" % (attempt, retries, rel, e))
                if attempt == retries:
                    raise
                time.sleep(5)


def verify_tree(src, dst):
    a, b = files_of(src), files_of(dst)
    a.pop("manifest.json", None)
    b.pop("manifest.json", None)
    missing = sorted(set(a) - set(b))
    wrong = sorted(k for k in a if k in b and a[k] != b[k])
    return missing, wrong


def drive_path(key):
    return Path(DRIVE_DIR) / Path(CHECKPOINTS[key]["handle"]).name


def store_to_drive(key, src):
    dst = drive_path(key)
    print("storing", key, "->", dst)
    copy_tree(src, dst)
    missing, wrong = verify_tree(src, dst)
    assert not missing and not wrong, f"verify failed: missing={missing} wrong_size={wrong}"
    (dst / "manifest.json").write_text(json.dumps({
        "handle": CHECKPOINTS[key]["handle"],
        "kaggle_version_dir": Path(src).name,
        "files": files_of(src),
        "stored_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }, indent=2))
    print("  verified %d files, %.1f GB" % (len(files_of(src)), dir_size(src) / 1e9))
    return dst


def restore_from_drive(key):
    src = drive_path(key)
    assert src.exists(), f"{src} not found - run the download and store cells first"
    dst = Path(LOCAL_MODELS) / src.name
    print("restoring", src, "->", dst)
    copy_tree(src, dst)
    missing, wrong = verify_tree(src, dst)
    assert not missing and not wrong, f"verify failed: missing={missing} wrong_size={wrong}"
    return dst

## 5. Install and download

In [ ]:
%pip install -q -U "kagglehub>=1.0"

In [ ]:
os.environ["KAGGLEHUB_CACHE"] = LOCAL_CACHE   # set before kagglehub is imported
import kagglehub

local_paths = {}
for key in FETCH:
    handle = CHECKPOINTS[key]["handle"]
    print("downloading", handle, "...")
    local_paths[key] = Path(kagglehub.model_download(handle))
    print("  ->", local_paths[key], "(%.1f GB)" % (dir_size(local_paths[key]) / 1e9))

## 6. Copy to Google Drive

In [ ]:
drive_paths = {key: store_to_drive(key, local_paths[key]) for key in FETCH}

## 7. Restore from Drive in a later session

Copies from Drive to the Colab disk, which is much faster to read from than Drive itself.
Needs sections 1, 3 and 4b to have been run, but no Kaggle login.

In [ ]:
RESTORE = "competition"   # which stored checkpoint to copy to the local disk
model_dir = restore_from_drive(RESTORE)
print(model_dir)

## 8. (Optional) Smoke test: load on the GPU and generate

This only checks that the stored checkpoint loads and answers. It is not the Phase 1 measurement
(memory at long context, tokens per second, thinking on/off); that is the next notebook.

The competition checkpoint is stored compressed (4-bit). Following the repo's `training/train_lora.py`,
it is loaded with `run_compressed=False`, which dequantizes to bf16 in memory. The other two checkpoints
load as ordinary bf16 weights. If loading fails with an unknown-architecture error, the installed
`transformers` is too old for Gemma 4; re-run the install cell and restart the runtime.

In [ ]:
%pip install -q -U transformers accelerate compressed-tensors

In [ ]:
import time, torch
from transformers import AutoModelForImageTextToText, AutoTokenizer, CompressedTensorsConfig

assert torch.cuda.is_available(), "Section 8 needs a GPU runtime (Runtime > Change runtime type)."

KEY = RESTORE
model_dir = restore_from_drive(KEY)

kwargs = dict(dtype=torch.bfloat16, device_map={"": 0})
if KEY == "competition":
    kwargs["quantization_config"] = CompressedTensorsConfig(run_compressed=False)

t0 = time.time()
model = AutoModelForImageTextToText.from_pretrained(model_dir, **kwargs)
tok = AutoTokenizer.from_pretrained(model_dir)
print("loaded in %.0fs; GPU memory %.1f GiB" % (time.time() - t0, torch.cuda.memory_allocated() / 2**30))

messages = [{"role": "user", "content": "Write a one-line Python function that reverses a string."}]
inputs = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)
t0 = time.time()
out = model.generate(**inputs, max_new_tokens=128, do_sample=False)
n_new = out.shape[-1] - inputs["input_ids"].shape[-1]
print("generated %d tokens in %.1fs (%.1f tok/s, includes warm-up)" % (n_new, time.time() - t0, n_new / (time.time() - t0)))
print(tok.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))
print("peak GPU memory %.1f GiB" % (torch.cuda.max_memory_allocated() / 2**30))